# Smart Expense Classifier - Complete ML Pipeline

**Dataset:** Hugging Face Transaction Categorization (4.5M records)

**Pipeline Steps:**
1. Exploratory Data Analysis (EDA)
2. Text Preprocessing
3. Baseline Model (TF-IDF + Logistic Regression)
4. Model Comparison (Naive Bayes, Linear SVM, Random Forest)
5. Evaluation (Accuracy, Precision, Recall, F1, Confusion Matrix)
6. Hyperparameter Tuning (GridSearchCV)
7. Model Saving & Prediction Function

---

## Setup & Imports

In [ ]:
# Standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
from collections import Counter
import joblib
import time

# Scikit-learn
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# NLTK for stop words
import nltk
try:
    from nltk.corpus import stopwords
except:
    nltk.download('stopwords')
    from nltk.corpus import stopwords

# Settings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)

# Random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(" Setup complete!")

## 1. Load Data

In [ ]:
# Load dataset (using sample for faster processing)
# For production, load full dataset: pd.read_csv('../data/huggingface_full_dataset.csv')

print("Loading dataset...")
start_time = time.time()

# Load sample (100k records for faster processing)
# Change this to load full dataset: nrows=None
df = pd.read_csv('../data/huggingface_full_dataset.csv', nrows=100000)

print(f" Loaded in {time.time() - start_time:.2f} seconds")
print(f"\nDataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print("="*70)
print("DATASET INFORMATION")
print("="*70)
print(f"Total records: {len(df):,}")
print(f"Features: {df.shape[1]}")
print(f"\nData types:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nUnique values:")
print(df.nunique())

### 2.1 Class Distribution

In [ ]:
# Category distribution
print("\nCategory Distribution:")
print("="*70)
category_counts = df['category'].value_counts()
for cat, count in category_counts.items():
    pct = (count / len(df)) * 100
    print(f"{cat:30s}: {count:6,} ({pct:5.2f}%)")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar plot
category_counts.plot(kind='barh', ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title('Category Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('Category')
axes[0].grid(alpha=0.3, axis='x')

# Pie chart
category_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Category Distribution (%)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('../reports/01_category_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n Category distribution analysis complete!")

### 2.2 Text Length Analysis

In [ ]:
# Add text length column
df['text_length'] = df['transaction_description'].str.len()
df['word_count'] = df['transaction_description'].str.split().str.len()

print("Text Length Statistics:")
print("="*70)
print(df[['text_length', 'word_count']].describe())

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Character length distribution
df['text_length'].hist(bins=50, ax=axes[0, 0], color='lightcoral', edgecolor='black')
axes[0, 0].set_title('Character Length Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Characters')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(df['text_length'].mean(), color='red', linestyle='--', label=f"Mean: {df['text_length'].mean():.1f}")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Word count distribution
df['word_count'].hist(bins=30, ax=axes[0, 1], color='lightgreen', edgecolor='black')
axes[0, 1].set_title('Word Count Distribution', fontweight='bold')
axes[0, 1].set_xlabel('Words')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].axvline(df['word_count'].mean(), color='green', linestyle='--', label=f"Mean: {df['word_count'].mean():.1f}")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Text length by category (boxplot)
df.boxplot(column='text_length', by='category', ax=axes[1, 0], rot=45)
axes[1, 0].set_title('Text Length by Category', fontweight='bold')
axes[1, 0].set_xlabel('Category')
axes[1, 0].set_ylabel('Characters')
plt.sca(axes[1, 0])
plt.xticks(rotation=45, ha='right')

# Word count by category (boxplot)
df.boxplot(column='word_count', by='category', ax=axes[1, 1], rot=45)
axes[1, 1].set_title('Word Count by Category', fontweight='bold')
axes[1, 1].set_xlabel('Category')
axes[1, 1].set_ylabel('Words')
plt.sca(axes[1, 1])
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig('../reports/02_text_length_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n Text length analysis complete!")

### 2.3 Common Words per Category

In [ ]:
# Get top words per category
def get_top_words(texts, n=10):
    """Extract top N words from text list"""
    all_words = []
    for text in texts:
        words = str(text).lower().split()
        all_words.extend(words)
    return Counter(all_words).most_common(n)

print("\nTop 10 Words per Category:")
print("="*70)

for category in df['category'].unique()[:5]:  # Show first 5 categories
    texts = df[df['category'] == category]['transaction_description']
    top_words = get_top_words(texts, n=10)
    print(f"\n{category}:")
    for word, count in top_words:
        print(f"  {word:20s}: {count:5,}")

print("\n Common words analysis complete!")

## 3. Text Preprocessing

In [ ]:
# Define preprocessing function
def preprocess_text(text):
    """
    Preprocess transaction description text:
    1. Convert to lowercase
    2. Remove punctuation and numbers
    3. Remove extra whitespace
    """
    if pd.isna(text):
        return ""
    
    # Convert to lowercase
    text = str(text).lower()
    
    # Remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

# Apply preprocessing
print("Preprocessing text...")
df['processed_text'] = df['transaction_description'].apply(preprocess_text)

# Show examples
print("\nPreprocessing Examples:")
print("="*70)
for i in range(5):
    print(f"Original : {df['transaction_description'].iloc[i]}")
    print(f"Processed: {df['processed_text'].iloc[i]}")
    print()

print(" Text preprocessing complete!")

## 4. Train-Test Split

In [ ]:
# Prepare features and labels
X = df['processed_text']
y = df['category']

# Split: 70% train, 15% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print("Dataset Split:")
print("="*70)
print(f"Total samples    : {len(X):,}")
print(f"Training samples : {len(X_train):,} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Validation samples: {len(X_val):,} ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test samples     : {len(X_test):,} ({len(X_test)/len(X)*100:.1f}%)")
print(f"\nClass distribution (train):")
print(y_train.value_counts())

print("\n Train-test split complete!")

## 5. Baseline Model: TF-IDF + Logistic Regression

In [ ]:
print("Training Baseline Model: TF-IDF + Logistic Regression")
print("="*70)

# Create pipeline
baseline_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.9,
        stop_words='english'
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

# Train
start_time = time.time()
baseline_pipeline.fit(X_train, y_train)
training_time = time.time() - start_time

print(f" Training completed in {training_time:.2f} seconds")

# Evaluate on all sets
train_score = baseline_pipeline.score(X_train, y_train)
val_score = baseline_pipeline.score(X_val, y_val)
test_score = baseline_pipeline.score(X_test, y_test)

print(f"\nBaseline Model Performance:")
print(f"Training Accuracy  : {train_score:.4f}")
print(f"Validation Accuracy: {val_score:.4f}")
print(f"Test Accuracy      : {test_score:.4f}")

# Detailed metrics on test set
y_pred = baseline_pipeline.predict(X_test)

print(f"\nDetailed Test Metrics:")
print(classification_report(y_test, y_pred, zero_division=0))

# Save baseline model
joblib.dump(baseline_pipeline, '../models/baseline_model_tfidf_logreg.joblib')
print("\n Baseline model saved!")

In [ ]:
# Confusion Matrix for baseline
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=baseline_pipeline.classes_)

fig, ax = plt.subplots(figsize=(12, 10))
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title('Confusion Matrix - Baseline Model (TF-IDF + Logistic Regression)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../reports/03_baseline_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(" Confusion matrix saved!")

## 6. Model Comparison

In [ ]:
print("Training Multiple Models for Comparison")
print("="*70)

# Define models to compare
models = {
    'Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')),
        ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1))
    ]),
    'Naive Bayes': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')),
        ('clf', MultinomialNB())
    ]),
    'Linear SVM': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')),
        ('clf', LinearSVC(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    'Random Forest': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=3000, ngram_range=(1, 2), stop_words='english')),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1))
    ])
}

# Store results
results = []

# Train and evaluate each model
for name, pipeline in models.items():
    print(f"\nTraining {name}...")
    
    # Train
    start_time = time.time()
    pipeline.fit(X_train, y_train)
    training_time = time.time() - start_time
    
    # Predict
    y_pred_train = pipeline.predict(X_train)
    y_pred_val = pipeline.predict(X_val)
    y_pred_test = pipeline.predict(X_test)
    
    # Calculate metrics
    train_acc = accuracy_score(y_train, y_pred_train)
    val_acc = accuracy_score(y_val, y_pred_val)
    test_acc = accuracy_score(y_test, y_pred_test)
    
    test_precision = precision_score(y_test, y_pred_test, average='weighted', zero_division=0)
    test_recall = recall_score(y_test, y_pred_test, average='weighted', zero_division=0)
    test_f1 = f1_score(y_test, y_pred_test, average='weighted', zero_division=0)
    
    # Store results
    results.append({
        'Model': name,
        'Train Acc': train_acc,
        'Val Acc': val_acc,
        'Test Acc': test_acc,
        'Precision': test_precision,
        'Recall': test_recall,
        'F1-Score': test_f1,
        'Time (s)': training_time
    })
    
    print(f"   Completed in {training_time:.2f}s - Test Accuracy: {test_acc:.4f}")

# Create results DataFrame
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Test Acc', ascending=False).reset_index(drop=True)

print("\n" + "="*70)
print("MODEL COMPARISON RESULTS")
print("="*70)
print(results_df.to_string(index=False))

# Save results
results_df.to_csv('../reports/model_comparison_results.csv', index=False)
print("\n Results saved to reports/model_comparison_results.csv")

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Test Accuracy comparison
results_df.plot(x='Model', y='Test Acc', kind='barh', ax=axes[0, 0], color='steelblue', legend=False)
axes[0, 0].set_title('Test Accuracy Comparison', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Accuracy')
axes[0, 0].grid(alpha=0.3, axis='x')

# Precision, Recall, F1 comparison
metrics_df = results_df[['Model', 'Precision', 'Recall', 'F1-Score']].set_index('Model')
metrics_df.plot(kind='barh', ax=axes[0, 1], width=0.8)
axes[0, 1].set_title('Precision, Recall, F1-Score Comparison', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Score')
axes[0, 1].legend(loc='lower right')
axes[0, 1].grid(alpha=0.3, axis='x')

# Training time comparison
results_df.plot(x='Model', y='Time (s)', kind='barh', ax=axes[1, 0], color='coral', legend=False)
axes[1, 0].set_title('Training Time Comparison', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Time (seconds)')
axes[1, 0].grid(alpha=0.3, axis='x')

# Train vs Val vs Test accuracy
acc_comparison = results_df[['Model', 'Train Acc', 'Val Acc', 'Test Acc']].set_index('Model')
acc_comparison.plot(kind='barh', ax=axes[1, 1], width=0.8)
axes[1, 1].set_title('Train / Validation / Test Accuracy', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Accuracy')
axes[1, 1].legend(loc='lower right')
axes[1, 1].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('../reports/04_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(" Model comparison visualization saved!")

## 7. Hyperparameter Tuning (Best Model)

In [ ]:
# Select best model (highest test accuracy)
best_model_name = results_df.iloc[0]['Model']
print(f"Best Model: {best_model_name}")
print("="*70)
print(f"Performing GridSearchCV for hyperparameter tuning...")

# Define parameter grid based on best model
if best_model_name == 'Logistic Regression':
    param_grid = {
        'tfidf__max_features': [3000, 5000, 7000],
        'tfidf__ngram_range': [(1, 1), (1, 2)],
        'clf__C': [0.1, 1.0, 10.0]
    }
    base_pipeline = models['Logistic Regression']
    
elif best_model_name == 'Linear SVM':
    param_grid = {
        'tfidf__max_features': [3000, 5000, 7000],
        'tfidf__ngram_range': [(1, 1), (1, 2)],
        'clf__C': [0.1, 1.0, 10.0]
    }
    base_pipeline = models['Linear SVM']
    
elif best_model_name == 'Naive Bayes':
    param_grid = {
        'tfidf__max_features': [3000, 5000, 7000],
        'tfidf__ngram_range': [(1, 1), (1, 2)],
        'clf__alpha': [0.1, 0.5, 1.0]
    }
    base_pipeline = models['Naive Bayes']
    
else:  # Random Forest
    param_grid = {
        'tfidf__max_features': [2000, 3000],
        'tfidf__ngram_range': [(1, 1), (1, 2)],
        'clf__n_estimators': [100, 200],
        'clf__max_depth': [None, 50]
    }
    base_pipeline = models['Random Forest']

# Perform GridSearchCV
grid_search = GridSearchCV(
    base_pipeline,
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

start_time = time.time()
grid_search.fit(X_train, y_train)
tuning_time = time.time() - start_time

print(f"\n GridSearchCV completed in {tuning_time:.2f} seconds")
print(f"\nBest Parameters:")
print(grid_search.best_params_)
print(f"\nBest Cross-Validation Score: {grid_search.best_score_:.4f}")

# Evaluate tuned model
tuned_model = grid_search.best_estimator_
test_score_tuned = tuned_model.score(X_test, y_test)
print(f"Test Accuracy (Tuned): {test_score_tuned:.4f}")

# Save tuned model
joblib.dump(tuned_model, '../models/best_model_tuned.joblib')
print("\n Tuned model saved as: models/best_model_tuned.joblib")

## 8. Final Evaluation & Confusion Matrix

In [ ]:
# Final predictions with tuned model
y_pred_final = tuned_model.predict(X_test)

print("FINAL MODEL EVALUATION")
print("="*70)
print(f"Model: {best_model_name} (Tuned)")
print(f"\nTest Set Metrics:")
print(f"Accuracy : {accuracy_score(y_test, y_pred_final):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_final, average='weighted', zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_final, average='weighted', zero_division=0):.4f}")
print(f"F1-Score : {f1_score(y_test, y_pred_final, average='weighted', zero_division=0):.4f}")

print(f"\nDetailed Classification Report:")
print(classification_report(y_test, y_pred_final, zero_division=0))

# Final confusion matrix
cm_final = confusion_matrix(y_test, y_pred_final)
disp_final = ConfusionMatrixDisplay(confusion_matrix=cm_final, display_labels=tuned_model.classes_)

fig, ax = plt.subplots(figsize=(14, 12))
disp_final.plot(ax=ax, cmap='Greens', values_format='d')
plt.title(f'Final Confusion Matrix - {best_model_name} (Tuned)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../reports/05_final_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n Final evaluation complete!")

## 9. Save Final Model & Create Prediction Function

In [ ]:
# Save the final model
final_model_path = '../models/final_expense_classifier.joblib'
joblib.dump(tuned_model, final_model_path)
print(f" Final model saved: {final_model_path}")

# Save metadata
metadata = {
    'model_name': best_model_name,
    'test_accuracy': float(test_score_tuned),
    'parameters': grid_search.best_params_,
    'categories': list(tuned_model.classes_),
    'training_samples': len(X_train),
    'random_state': RANDOM_STATE
}

import json
with open('../models/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(" Model metadata saved")

In [ ]:
# Create prediction function
def predict(text, model_path='../models/final_expense_classifier.joblib'):
    """
    Predict category and confidence for a transaction description
    
    Args:
        text (str): Transaction description
        model_path (str): Path to saved model
    
    Returns:
        dict: Prediction results with category and confidence
    """
    # Load model
    model = joblib.load(model_path)
    
    # Preprocess
    processed = preprocess_text(text)
    
    # Predict
    prediction = model.predict([processed])[0]
    probabilities = model.predict_proba([processed])[0]
    confidence = max(probabilities)
    
    # Get top 3 predictions
    top_3_idx = np.argsort(probabilities)[-3:][::-1]
    top_3_categories = [(model.classes_[i], probabilities[i]) for i in top_3_idx]
    
    return {
        'category': prediction,
        'confidence': float(confidence),
        'top_3': top_3_categories
    }

# Test prediction function
print("Testing Prediction Function:")
print("="*70)

test_examples = [
    "Starbucks Coffee",
    "Uber ride to airport",
    "Amazon purchase electronics",
    "Doctor appointment",
    "Electric bill payment"
]

for example in test_examples:
    result = predict(example, model_path=final_model_path)
    print(f"\nText: {example}")
    print(f"Category: {result['category']}")
    print(f"Confidence: {result['confidence']:.2%}")
    print(f"Top 3: {[(cat, f'{prob:.2%}') for cat, prob in result['top_3']]}")

print("\n Prediction function working correctly!")

## 10. Summary & Next Steps

In [ ]:
print("\n" + "="*70)
print("PROJECT SUMMARY")
print("="*70)
print(f"\n Dataset: {len(df):,} records processed")
print(f" Best Model: {best_model_name}")
print(f" Test Accuracy: {test_score_tuned:.4f}")
print(f" Models Saved:")
print(f"   - Baseline: models/baseline_model_tfidf_logreg.joblib")
print(f"   - Final: models/final_expense_classifier.joblib")
print(f" Reports Generated: {len([f for f in os.listdir('../reports') if f.endswith('.png')])} visualizations")
print(f"\n Next Steps:")
print(f"   1. Create Streamlit demo app")
print(f"   2. Test with real transaction data")
print(f"   3. Deploy model to production")
print(f"   4. Monitor performance metrics")
print("\n" + "="*70)
print(" PIPELINE COMPLETE!")
print("="*70)